In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

In [ ]:
# Construct the raw GitHub URL for the .csv.gz file
github_url = 'https://raw.githubusercontent.com/Break-Through-Tech/Healthcare-1B-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation/main/data/kidney_waitlist_analytic.csv.gz'

# Load the gzipped CSV file directly into a pandas DataFrame
kidney_waitlist_analytics = pd.read_csv(github_url, compression='gzip')

# Display the first 5 rows of the DataFrame
display(kidney_waitlist_analytics.head())

/tmp/ipykernel_2256/174966764.py:5: DtypeWarning: Columns (22,23,24,26,27,30) have mixed types. Specify dtype option on import or set low_memory=False.
  kidney_waitlist_analytics = pd.read_csv(github_url, compression='gzip')


,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,REM_CD,...,PSTATUS,PTIME,TRR_ID_CODE,DONOR_ID,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,NaN,24.0,...,NaN,NaN,NaN,NaN,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,18352,removed_too_sick,1,0,0,231.0


## **Task 3: Data Missingness & Outliers**
*Done by: Valarie*






In [ ]:
# Remove columns with 90%+ missing values
# MULTIORG is 98% null, A2A2B_ELIGIBILITY is 92% null — not usable

cols_to_drop = ["MULTIORG", "A2A2B_ELIGIBILITY"]
df = kidney_waitlist_analytics.copy()
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("Dropped columns:", cols_to_drop)

Dropped columns: ['MULTIORG', 'A2A2B_ELIGIBILITY']


In [ ]:
# These 7 columns are only populated when a transplant actually happens
# So they should be either ALL missing or ALL present for a given row
structural_cols = ["PREV_TX", "PSTATUS", "TRR_ID_CODE", "DONOR_ID",
                   "DON_TY", "TX_DATE", "ORGAN"]
missing_count = df[structural_cols].isna().sum(axis=1)
print(missing_count.value_counts())
# Expected: ~264k rows have all 7 present, ~231k missing all 7

7    264107
0    230755
Name: count, dtype: int64


In [ ]:
# These 7 columns should ONLY be filled after a transplant actually happens
# So they should be either ALL missing or ALL present for a given row
structural_cols = ["PREV_TX", "PSTATUS", "TRR_ID_CODE", "DONOR_ID",
                   "DON_TY", "TX_DATE", "ORGAN"]
missing_count = df[structural_cols].isna().sum(axis=1)
print(missing_count.value_counts()) # finding missingness in these columns here
# Expected: ~264k rows have all 7 present, ~231k missing all 7

7    264107
0    230755
Name: count, dtype: int64


In [ ]:
# Look for impossible values (negative ages, extreme BMIs, negative durations, etc...)
numeric_cols = ["INIT_AGE", "BMI_TCR", "DAYSWAIT_ALLOC", "DAYSWAIT_CHRON", "days_to_event"]
for col in numeric_cols:
    if col in df.columns:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}, negatives={(df[col] < 0).sum()}")
        # show the min and max values of each column, which made me discover the high bmi score

INIT_AGE: min=0, max=91, negatives=0
BMI_TCR: min=0.18, max=430226.67, negatives=0
DAYSWAIT_ALLOC: min=0.0, max=15395.0, negatives=0
DAYSWAIT_CHRON: min=0.0, max=4200.0, negatives=0
days_to_event: min=0.0, max=4200.0, negatives=0


In [ ]:
# Flag rows where END_DATE < INIT_DATE since it would make no sense
exclude_mask = pd.Series(False, index=df.index)
for col in ["DAYSWAIT_ALLOC", "DAYSWAIT_CHRON", "days_to_event"]:
    if col in df.columns:
        exclude_mask = exclude_mask | (df[col] < 0)
        # discover remove rows with negative days

print(f"Rows with negative durations: {exclude_mask.sum()}")
flagged_rows = df[exclude_mask]
flagged_rows.to_csv("excluded_rows_log.csv", index=False)
        # discover remove rows with negative days

Rows with negative durations: 0


In [ ]:
# Find the outliers BMI below 10 or above 100 since that is medically unrealistic
bmi_outliers = (df["BMI_TCR"] < 10) | (df["BMI_TCR"] > 100)

print(f"BMI outliers: {bmi_outliers.sum()}")
print(df.loc[bmi_outliers, "BMI_TCR"].describe())

# Combine with the negative-duration flags for a total exclusion count to get potential rows to excluded
exclude_mask = exclude_mask | bmi_outliers
print(f"Total flagged rows: {exclude_mask.sum()}")

flagged_rows = df[exclude_mask]
flagged_rows.to_csv("excluded_rows_log.csv", index=False) #export

BMI outliers: 503
count       503.000000
mean      35196.955765
std       91965.523768
min           0.180000
25%           7.500000
50%         120.720000
75%         835.630000
max      430226.670000
Name: BMI_TCR, dtype: float64
Total flagged rows: 503


In [ ]:
# null bmi scores, didn't delete. Instead, set outlier BMIs to NaN so the rows are still usable for other features
# This is better than dropping entire rows
df.loc[bmi_outliers, "BMI_TCR"] = np.nan
print(f"Nulled out {bmi_outliers.sum()} bad BMI values")
print(f"BMI_TCR missing % now: {df['BMI_TCR'].isna().mean() * 100:.2f}%")
# Missingness went from 0.31% => 0.42%

Nulled out 503 bad BMI values
BMI_TCR missing % now: 0.42%


## Task #**1: Standardize Categorical Variables, Normalize Numerical Features**
Done by Johnny



In [ ]:
# Task #1: Standardize categorical variables, normalize numerical features
# Only listing-time features are used (no leaky/transplant-time columns)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Listing-time categorical features
categorical_features = ["GENDER", "ETHCAT", "ABO", "ON_DIALYSIS",
                         "FUNC_STAT_TCR", "INIT_STAT", "REGION"]

# Listing-time numerical features
numerical_features = ["INIT_AGE", "INIT_CPRA", "BMI_TCR"]

# Build the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

# Fit and transform the feature set
X = df[categorical_features + numerical_features]
X_transformed = preprocessor.fit_transform(X)

print("Shape before transform:", X.shape)
print("Shape after transform:", X_transformed.shape)


Shape before transform: (494862, 10)
Shape after transform: (494862, 63)
